# HemaFAIR Registry OMOP ETL — Exercise

In this exercise you will populate your own OMOP CDM database with synthetic HemaFAIR registry data.

**What is already done for you:**
- Database connection and helper functions
- Loading the source CSVs into an `import` schema
- The `PERSON` table — fully worked with explanations
- The `OBSERVATION_PERIOD` table — fully worked as a second example

**What you need to fill in (marked with `# TODO`):**
- Concept IDs for VISIT_OCCURRENCE, CONDITION_OCCURRENCE, OBSERVATION, MEASUREMENT, PROCEDURE_OCCURRENCE
- CASE statement mappings for diagnoses
- HGNC gene identifiers for genetic allele observations

**Tip:** Use ATHENA (https://athena.ohdsi.org) and the concept IDs you found in Session 3.

Run each cell in order. After each section there is a verification query — check the row counts match expectations.

## Connection Settings

Replace `trainee_01` with your trainee number and fill in your password.

In [ ]:
DB_HOST     = "localhost"
DB_PORT     = 5432
DB_NAME     = "trainee_01"   # replace with your trainee number
DB_USER     = "trainee_01"   # replace with your trainee number
DB_PASSWORD = ""             # fill in your password

print(f"Connecting to {DB_USER}@{DB_HOST}:{DB_PORT}/{DB_NAME}")

## Setup

In [ ]:
import psycopg2
import pandas as pd
from sqlalchemy import create_engine, text
from urllib.parse import quote_plus

con = psycopg2.connect(
    host=DB_HOST, port=DB_PORT,
    dbname=DB_NAME, user=DB_USER, password=DB_PASSWORD
)
con.autocommit = True

engine = create_engine(
    f"postgresql+psycopg2://{DB_USER}:{quote_plus(DB_PASSWORD)}@{DB_HOST}:{DB_PORT}/{DB_NAME}"
)

def execute(sql):
    with con.cursor() as cur:
        cur.execute(sql)

def query(sql):
    with engine.connect() as conn:
        return pd.read_sql_query(text(sql), conn)

print("Connected to PostgreSQL.")

## Explore: OMOP CDM Tables

Run this to see all tables that exist in your OMOP schema. Your database was pre-loaded with the CDM structure and the OMOP vocabulary.

In [ ]:
query("""
SELECT table_name
FROM information_schema.tables
WHERE table_schema = 'omop'
ORDER BY table_name
""")

## Fill CDM Source

The `cdm_source` table records metadata about this dataset. Run as-is.

In [ ]:
execute("""
INSERT INTO omop.cdm_source (
    cdm_source_name, cdm_source_abbreviation, cdm_holder,
    source_release_date, cdm_release_date,
    cdm_version_concept_id, vocabulary_version
)
VALUES (
    'HemaFAIR', 'HemaFAIR', 'HemaFAIR',
    '2026-05-06', '2026-05-06',
    756265,  -- CDM version 5.4
    'v5.0 27-FEB-25'
);
""")

query("SELECT * FROM omop.cdm_source")

## Import HemaFAIR Registry Data

The registry data has already been loaded into the  schema for you.
Run the cell below to preview the three tables.


In [ ]:
# The import schema is pre-populated — no action needed.
# Run the cell below to preview the data.


In [ ]:
print("--- dictionary ---")
display(query("SELECT * FROM import.dictionary LIMIT 5"))

print("--- mappings ---")
display(query("SELECT * FROM import.mappings LIMIT 5"))

print("--- labels ---")
display(query("SELECT * FROM import.labels LIMIT 5"))

---
## ETL: Fill OMOP Tables

### PERSON ✅ (worked example — run as-is)

Study this cell carefully. It shows the patterns used throughout the rest of the exercise:
- `row_number() OVER ()` generates a unique integer ID
- `CASE ... WHEN ... THEN ... ELSE 0 END` maps source values to concept IDs
- `EXTRACT(YEAR FROM date_column)::integer` extracts date parts in PostgreSQL
- The source value (`"Patient's sex at birth"`) is preserved in `gender_source_value` alongside the concept ID
- `person_source_value` stores the original Record ID so we can join back later

In [ ]:
execute("""
INSERT INTO omop.person (
    person_id, gender_concept_id,
    year_of_birth, month_of_birth, day_of_birth,
    race_concept_id, ethnicity_concept_id,
    person_source_value, gender_source_value
)
SELECT
    row_number() OVER () AS person_id,   -- generate a surrogate key
    CASE "Patient's sex at birth"         -- map source text to OMOP concept ID
        WHEN 'Male'   THEN 8507          -- SNOMED: Male
        WHEN 'Female' THEN 8532          -- SNOMED: Female
        ELSE 0                           -- unknown / unmapped
    END,
    EXTRACT(YEAR  FROM "Patient's date of birth")::integer,
    EXTRACT(MONTH FROM "Patient's date of birth")::integer,
    EXTRACT(DAY   FROM "Patient's date of birth")::integer,
    0,   -- race_concept_id: not collected in source
    0,   -- ethnicity_concept_id: not collected in source
    "Record ID",             -- stored as person_source_value for joining later
    "Patient's sex at birth" -- preserved for audit
FROM import.labels;
""")

query("SELECT * FROM omop.person")

### OBSERVATION_PERIOD ✅ (worked example — run as-is)

No recording date is available in the HemaFAIR registry data, so we assign a single date (2026-05-12) as a pragmatic proxy.
This is common in registry data. `period_type_concept_id = 32809` means "Case Report Form".

In [ ]:
execute("""
INSERT INTO omop.observation_period (
    observation_period_id, person_id,
    observation_period_start_date, observation_period_end_date,
    period_type_concept_id
)
SELECT
    row_number() OVER (),
    person_id,
    '2026-05-12',
    '2026-05-12',
    32809  -- Case Report Form
FROM omop.person;
""")

query("SELECT * FROM omop.observation_period")

### VISIT_OCCURRENCE 📝

Fill in the two missing concept IDs:

1. **`visit_concept_id`**: what type of visit is this? Each patient has one registry encounter.  
   Search ATHENA for "Home visit" (Domain: Visit) — or look for another visit type that fits a registry encounter.

2. **`visit_type_concept_id`**: how was this visit recorded?  
   Hint: it was captured on a Case Report Form. The same concept is used in OBSERVATION_PERIOD above.

In [ ]:
execute("""
INSERT INTO omop.visit_occurrence (
    visit_occurrence_id, person_id, visit_concept_id,
    visit_start_date, visit_start_datetime,
    visit_end_date, visit_end_datetime,
    visit_type_concept_id
)
SELECT
    row_number() OVER (),
    p.person_id,
    0,     -- TODO: find the concept ID for the visit type (e.g. "Home visit")
    op.observation_period_start_date,
    op.observation_period_start_date,
    op.observation_period_end_date,
    op.observation_period_end_date,
    0      -- TODO: concept ID for how this visit was recorded (hint: same as observation_period above)
FROM omop.person p
JOIN omop.observation_period op ON op.person_id = p.person_id;
""")

query("SELECT * FROM omop.visit_occurrence LIMIT 5")

### CONDITION_OCCURRENCE 📝

We insert conditions in multiple batches, using a **sequence** to keep IDs unique across batches.

**Part 1 — Primary diagnosis:**  
Each patient has one primary diagnosis. Fill in the SNOMED concept IDs for each diagnosis value.  
You found Beta-thalassaemia in Session 3 — that one is already filled in as an example.

| Diagnosis | concept_id |
|-----------|------------|
| Beta-thalassaemia | 65959000 ✅ |
| Alpha-thalassaemia | ? |
| Sickle cell disease | ? |

In [ ]:
execute("CREATE SEQUENCE condition_id_seq START 1;")

# Primary diagnosis
execute("""
INSERT INTO omop.condition_occurrence (
    condition_occurrence_id, person_id, condition_concept_id,
    condition_start_date, condition_start_datetime,
    condition_end_date, condition_end_datetime,
    condition_type_concept_id, visit_occurrence_id, condition_source_value
)
SELECT
    nextval('condition_id_seq'),
    p.person_id,
    CASE "Diagnosis retained by the specialised centre"
        WHEN 'Alpha-thalassemia'   THEN 0       -- TODO: find SNOMED concept ID
        WHEN 'Beta-thalassemia'    THEN 65959000 -- SNOMED: Beta-thalassemia
        WHEN 'Sickle cell disease' THEN 0       -- TODO: find SNOMED concept ID
        ELSE 0
    END,
    vo.visit_start_date, vo.visit_start_date,
    vo.visit_start_date, vo.visit_start_date,
    32865,  -- Patient self report
    vo.visit_occurrence_id,
    "Diagnosis retained by the specialised centre"
FROM import.labels l
JOIN omop.person p ON l."Record ID" = p.person_source_value
JOIN omop.visit_occurrence vo ON p.person_id = vo.person_id;
""")

query("SELECT condition_source_value, condition_concept_id, COUNT(*) FROM omop.condition_occurrence GROUP BY 1, 2 ORDER BY 1")

**Part 2 — Comorbidities:**

The three examples below (Hypopituitarism, Infertility, Hepatitis) are already filled in — study the pattern.  
Fill in the concept IDs for the three conditions marked `TODO`.

| Condition | concept_id |
|-----------|------------|
| Hypopituitarism | 4254542 ✅ |
| Infertile | 4311387 ✅ |
| Acute viral hepatitis | 4211974 ✅ |
| Vitamin D deficiency | ? |
| Osteoporosis | ? |
| Osteopenia | ? |
| Acute chest syndrome | 254062 ✅ |

In [ ]:
# Hypopituitarism ✅ (example)
execute("""
INSERT INTO omop.condition_occurrence (
    condition_occurrence_id, person_id, condition_concept_id,
    condition_start_date, condition_start_datetime,
    condition_end_date, condition_end_datetime,
    condition_type_concept_id, visit_occurrence_id, condition_source_value
)
SELECT nextval('condition_id_seq'), p.person_id,
    4254542,  -- Hypopituitarism
    vo.visit_start_date, vo.visit_start_date,
    vo.visit_start_date, vo.visit_start_date,
    32865, vo.visit_occurrence_id, l."Hypopituitarism"
FROM import.labels l
JOIN omop.person p ON l."Record ID" = p.person_source_value
JOIN omop.visit_occurrence vo ON p.person_id = vo.person_id
WHERE l."Hypopituitarism" = 'Yes';
""")

# Infertile ✅ (example)
execute("""
INSERT INTO omop.condition_occurrence (
    condition_occurrence_id, person_id, condition_concept_id,
    condition_start_date, condition_start_datetime,
    condition_end_date, condition_end_datetime,
    condition_type_concept_id, visit_occurrence_id, condition_source_value
)
SELECT nextval('condition_id_seq'), p.person_id,
    4311387,  -- Infertile
    vo.visit_start_date, vo.visit_start_date,
    vo.visit_start_date, vo.visit_start_date,
    32865, vo.visit_occurrence_id, l."Is the patient infertile?"
FROM import.labels l
JOIN omop.person p ON l."Record ID" = p.person_source_value
JOIN omop.visit_occurrence vo ON p.person_id = vo.person_id
WHERE l."Is the patient infertile?" = 'Yes';
""")

# Acute viral hepatitis ✅ (example)
execute("""
INSERT INTO omop.condition_occurrence (
    condition_occurrence_id, person_id, condition_concept_id,
    condition_start_date, condition_start_datetime,
    condition_end_date, condition_end_datetime,
    condition_type_concept_id, visit_occurrence_id, condition_source_value
)
SELECT nextval('condition_id_seq'), p.person_id,
    4211974,  -- Acute viral hepatitis
    vo.visit_start_date, vo.visit_start_date,
    vo.visit_start_date, vo.visit_start_date,
    32865, vo.visit_occurrence_id, l."Acute viral hepatitis"
FROM import.labels l
JOIN omop.person p ON l."Record ID" = p.person_source_value
JOIN omop.visit_occurrence vo ON p.person_id = vo.person_id
WHERE l."Acute viral hepatitis" = 'Yes';
""")

# Vitamin D deficiency 📝 TODO
execute("""
INSERT INTO omop.condition_occurrence (
    condition_occurrence_id, person_id, condition_concept_id,
    condition_start_date, condition_start_datetime,
    condition_end_date, condition_end_datetime,
    condition_type_concept_id, visit_occurrence_id, condition_source_value
)
SELECT nextval('condition_id_seq'), p.person_id,
    0,  -- TODO: find the SNOMED concept ID for Vitamin D deficiency
    vo.visit_start_date, vo.visit_start_date,
    vo.visit_start_date, vo.visit_start_date,
    32865, vo.visit_occurrence_id, l."Vitamin D deficiency"
FROM import.labels l
JOIN omop.person p ON l."Record ID" = p.person_source_value
JOIN omop.visit_occurrence vo ON p.person_id = vo.person_id
WHERE l."Vitamin D deficiency" = 'Yes, confirmed';
""")

# Osteoporosis 📝 TODO
execute("""
INSERT INTO omop.condition_occurrence (
    condition_occurrence_id, person_id, condition_concept_id,
    condition_start_date, condition_start_datetime,
    condition_end_date, condition_end_datetime,
    condition_type_concept_id, visit_occurrence_id, condition_source_value
)
SELECT nextval('condition_id_seq'), p.person_id,
    0,  -- TODO: find the SNOMED concept ID for Osteoporosis
    vo.visit_start_date, vo.visit_start_date,
    vo.visit_start_date, vo.visit_start_date,
    32865, vo.visit_occurrence_id, l."Osteoporosis"
FROM import.labels l
JOIN omop.person p ON l."Record ID" = p.person_source_value
JOIN omop.visit_occurrence vo ON p.person_id = vo.person_id
WHERE l."Osteoporosis" = 'Yes, confirmed';
""")

# Osteopenia 📝 TODO
execute("""
INSERT INTO omop.condition_occurrence (
    condition_occurrence_id, person_id, condition_concept_id,
    condition_start_date, condition_start_datetime,
    condition_end_date, condition_end_datetime,
    condition_type_concept_id, visit_occurrence_id, condition_source_value
)
SELECT nextval('condition_id_seq'), p.person_id,
    0,  -- TODO: find the SNOMED concept ID for Osteopenia
    vo.visit_start_date, vo.visit_start_date,
    vo.visit_start_date, vo.visit_start_date,
    32865, vo.visit_occurrence_id, l."Osteopenia"
FROM import.labels l
JOIN omop.person p ON l."Record ID" = p.person_source_value
JOIN omop.visit_occurrence vo ON p.person_id = vo.person_id
WHERE l."Osteopenia" = 'Yes, confirmed';
""")

# Acute chest syndrome ✅ (example)
execute("""
INSERT INTO omop.condition_occurrence (
    condition_occurrence_id, person_id, condition_concept_id,
    condition_start_date, condition_start_datetime,
    condition_end_date, condition_end_datetime,
    condition_type_concept_id, visit_occurrence_id, condition_source_value
)
SELECT nextval('condition_id_seq'), p.person_id,
    254062,  -- Acute chest syndrome
    vo.visit_start_date, vo.visit_start_date,
    vo.visit_start_date, vo.visit_start_date,
    32865, vo.visit_occurrence_id, l."Acute chest syndrome"
FROM import.labels l
JOIN omop.person p ON l."Record ID" = p.person_source_value
JOIN omop.visit_occurrence vo ON p.person_id = vo.person_id
WHERE l."Acute chest syndrome" = 'Yes, confirmed';
""")

query("SELECT condition_source_value, condition_concept_id, COUNT(*) AS n FROM omop.condition_occurrence GROUP BY 1, 2 ORDER BY 3 DESC")

### OBSERVATION 📝

Observations capture clinical findings that don't fit into conditions or measurements: country of birth, transfusion status, genetic alleles, drug compliance.

**Part 1 — Country of birth:**  
Australia and Canada are filled in as examples. Fill in the missing concept IDs for Congo, Cyprus, Greece, and Iraq.
Search ATHENA for the country name (Domain: Observation or Condition; Vocabulary: SNOMED).

| Country | concept_id |
|---------|------------|
| Australia | 4199969 ✅ |
| Canada | 4200105 ✅ |
| Congo | ? |
| Cyprus | ? |
| Greece | ? |
| Iraq | ? |
| Syria | 4153306 ✅ |
| United Kingdom | 4202086 ✅ |
| Unknown | 40482029 ✅ |

In [ ]:
execute("CREATE SEQUENCE observation_id_seq START 1;")

# Country of birth 📝
execute("""
INSERT INTO omop.observation (
    observation_id, person_id, observation_concept_id,
    observation_date, observation_datetime,
    observation_type_concept_id, visit_occurrence_id, value_source_value
)
SELECT
    nextval('observation_id_seq'), p.person_id,
    CASE l."Patient's country of birth"
        WHEN 'Australia'      THEN 4199969  -- ✅
        WHEN 'Canada'         THEN 4200105  -- ✅
        WHEN 'Congo'          THEN 0        -- TODO: find SNOMED concept for Congo
        WHEN 'Cyprus'         THEN 0        -- TODO: find SNOMED concept for Cyprus
        WHEN 'Greece'         THEN 0        -- TODO: find SNOMED concept for Greece
        WHEN 'Iraq'           THEN 0        -- TODO: find SNOMED concept for Iraq
        WHEN 'Syria'          THEN 4153306  -- ✅
        WHEN 'United Kingdom' THEN 4202086  -- ✅
        ELSE 40482029  -- Country of birth unknown
    END,
    vo.visit_start_date, vo.visit_start_date,
    32865,
    vo.visit_occurrence_id,
    l."Patient's country of birth"
FROM import.labels l
JOIN omop.person p ON l."Record ID" = p.person_source_value
JOIN omop.visit_occurrence vo ON p.person_id = vo.person_id;
""")

**Part 2 — Transfusion status and drug compliance:** ✅ (run as-is)

These are given as examples. Note how drug compliance is split into two separate observations depending on the category.

In [ ]:
# Transfusion status ✅
execute("""
INSERT INTO omop.observation (
    observation_id, person_id, observation_concept_id,
    observation_date, observation_datetime,
    observation_type_concept_id, visit_occurrence_id, value_source_value
)
SELECT
    nextval('observation_id_seq'), p.person_id,
    40758326,  -- Transfusion status qualitative
    vo.visit_start_date, vo.visit_start_date, 32865, vo.visit_occurrence_id,
    l."Does the patient require regular or occasional transfusions in the present (in the last 12 months) ?"
FROM import.labels l
JOIN omop.person p ON l."Record ID" = p.person_source_value
JOIN omop.visit_occurrence vo ON p.person_id = vo.person_id;
""")

# Drug compliance — poor ✅
execute("""
INSERT INTO omop.observation (
    observation_id, person_id, observation_concept_id,
    observation_date, observation_datetime,
    observation_type_concept_id, visit_occurrence_id, value_source_value
)
SELECT nextval('observation_id_seq'), p.person_id,
    4292063,  -- Drug compliance poor
    vo.visit_start_date, vo.visit_start_date, 32865,
    vo.visit_occurrence_id, l."Drug compliance"
FROM import.labels l
JOIN omop.person p ON l."Record ID" = p.person_source_value
JOIN omop.visit_occurrence vo ON p.person_id = vo.person_id
WHERE l."Drug compliance" = 'Poor (< 60% of recommended dose)';
""")

# Drug compliance — good / excellent ✅
execute("""
INSERT INTO omop.observation (
    observation_id, person_id, observation_concept_id,
    observation_date, observation_datetime,
    observation_type_concept_id, visit_occurrence_id, value_source_value
)
SELECT nextval('observation_id_seq'), p.person_id,
    4056965,  -- Drug compliance good
    vo.visit_start_date, vo.visit_start_date, 32865,
    vo.visit_occurrence_id, l."Drug compliance"
FROM import.labels l
JOIN omop.person p ON l."Record ID" = p.person_source_value
JOIN omop.visit_occurrence vo ON p.person_id = vo.person_id
WHERE l."Drug compliance" = 'Good (80-60 % of recommended dose)'
   OR l."Drug compliance" = 'Excellent (>80% of recommended dose)';
""")

**Part 3 — Genetic alleles 📝**

HBB Allele 1 is given as a full example — note:
- `observation_concept_id = 0`: no OMOP standard concept exists for specific allele variants
- `value_as_string`: stores the actual allele value ("b+", "b0", etc.)
- `value_source_value`: stores the HGNC gene identifier for the gene this allele belongs to

Fill in the `value_source_value` for HBB Allele 2 (same gene as Allele 1).  
For HBA Allele 1 and 2: the HBA genes are HBA1 and HBA2 — look up their HGNC identifiers.

In [ ]:
# HBB Allele 1 ✅ (full example)
execute("""
INSERT INTO omop.observation (
    observation_id, person_id, observation_concept_id,
    observation_date, observation_datetime,
    observation_type_concept_id, value_as_string,
    visit_occurrence_id, value_source_value
)
SELECT nextval('observation_id_seq'), p.person_id,
    0,  -- No OMOP concept for specific HBB allele variants
    vo.visit_start_date, vo.visit_start_date, 32865,
    l."HBB Allele 1",   -- the allele value (b+, b0, ...)
    vo.visit_occurrence_id,
    'HGNC:4827'  -- HGNC identifier for the HBB gene
FROM import.labels l
JOIN omop.person p ON l."Record ID" = p.person_source_value
JOIN omop.visit_occurrence vo ON p.person_id = vo.person_id;
""")

# HBB Allele 2 📝
execute("""
INSERT INTO omop.observation (
    observation_id, person_id, observation_concept_id,
    observation_date, observation_datetime,
    observation_type_concept_id, value_as_string,
    visit_occurrence_id, value_source_value
)
SELECT nextval('observation_id_seq'), p.person_id,
    0,
    vo.visit_start_date, vo.visit_start_date, 32865,
    l."HBB Allele 2",
    vo.visit_occurrence_id,
    'TODO'  -- TODO: same gene as HBB Allele 1 — what is the HGNC ID?
FROM import.labels l
JOIN omop.person p ON l."Record ID" = p.person_source_value
JOIN omop.visit_occurrence vo ON p.person_id = vo.person_id;
""")

# HBA Allele 1 📝
execute("""
INSERT INTO omop.observation (
    observation_id, person_id, observation_concept_id,
    observation_date, observation_datetime,
    observation_type_concept_id, value_as_string,
    visit_occurrence_id, value_source_value
)
SELECT nextval('observation_id_seq'), p.person_id,
    0,
    vo.visit_start_date, vo.visit_start_date, 32865,
    l."HBA Allele 1",
    vo.visit_occurrence_id,
    'TODO'  -- TODO: look up HGNC identifiers for HBA1 and HBA2 genes (format: 'HGNC:xxxx;HGNC:xxxx')
FROM import.labels l
JOIN omop.person p ON l."Record ID" = p.person_source_value
JOIN omop.visit_occurrence vo ON p.person_id = vo.person_id;
""")

# HBA Allele 2 📝
execute("""
INSERT INTO omop.observation (
    observation_id, person_id, observation_concept_id,
    observation_date, observation_datetime,
    observation_type_concept_id, value_as_string,
    visit_occurrence_id, value_source_value
)
SELECT nextval('observation_id_seq'), p.person_id,
    0,
    vo.visit_start_date, vo.visit_start_date, 32865,
    l."HBA Allele 2",
    vo.visit_occurrence_id,
    'TODO'  -- TODO: same as HBA Allele 1
FROM import.labels l
JOIN omop.person p ON l."Record ID" = p.person_source_value
JOIN omop.visit_occurrence vo ON p.person_id = vo.person_id;
""")

query("SELECT COUNT(*) AS observation_rows FROM omop.observation")

### MEASUREMENT 📝

Measurements are numeric values with a unit. Fill in the missing concept IDs.

- **Cardiac iron T2\*** and **Liver MRI T2\***: no standard concept exists → `concept_id = 0`. But we still need a unit.  
  The unit is milliseconds — find its OMOP unit concept ID in ATHENA (Domain: Unit).

- **Ferritin serum**: a standard LOINC concept exists. Find `measurement_concept_id` and the concept ID for `ng/mL` (Domain: Unit).

Cardiac T2\* and Cirrhosis are filled in as examples.

In [ ]:
execute("CREATE SEQUENCE measurement_id_seq START 1;")

# Cardiac iron T2* — no OMOP concept, but we know the unit ✅ (partial example)
execute("""
INSERT INTO omop.measurement (
    measurement_id, person_id, measurement_concept_id,
    measurement_date, measurement_datetime,
    measurement_type_concept_id, value_as_number,
    unit_concept_id, visit_occurrence_id, value_source_value
)
SELECT nextval('measurement_id_seq'), p.person_id,
    0,  -- No suitable OMOP concept for this MRI parameter
    vo.visit_start_date, vo.visit_start_date, 32809,
    l."Cardiac iron T2*  (milisec)",
    9593,  -- millisecond ✅
    vo.visit_occurrence_id, 'Cardiac iron T2'
FROM import.labels l
JOIN omop.person p ON l."Record ID" = p.person_source_value
JOIN omop.visit_occurrence vo ON p.person_id = vo.person_id
WHERE l."Cardiac iron T2*  (milisec)" IS NOT NULL;
""")

# Liver MRI T2* 📝
execute("""
INSERT INTO omop.measurement (
    measurement_id, person_id, measurement_concept_id,
    measurement_date, measurement_datetime,
    measurement_type_concept_id, value_as_number,
    unit_concept_id, visit_occurrence_id, value_source_value
)
SELECT nextval('measurement_id_seq'), p.person_id,
    0,
    vo.visit_start_date, vo.visit_start_date, 32809,
    l."Liver MRI T2* (milisec)",
    0,  -- TODO: same unit as Cardiac T2* — what is the concept ID for millisecond?
    vo.visit_occurrence_id, 'Liver MRI T2'
FROM import.labels l
JOIN omop.person p ON l."Record ID" = p.person_source_value
JOIN omop.visit_occurrence vo ON p.person_id = vo.person_id
WHERE l."Liver MRI T2* (milisec)" IS NOT NULL;
""")

# Cirrhosis ✅ (example — note: no value_as_number, just presence)
execute("""
INSERT INTO omop.measurement (
    measurement_id, person_id, measurement_concept_id,
    measurement_date, measurement_datetime,
    measurement_type_concept_id, unit_concept_id,
    visit_occurrence_id, value_source_value
)
SELECT nextval('measurement_id_seq'), p.person_id,
    36770062,  -- Cirrhosis ✅
    vo.visit_start_date, vo.visit_start_date, 32809,
    9593, vo.visit_occurrence_id, l."Cirrhosis"
FROM import.labels l
JOIN omop.person p ON l."Record ID" = p.person_source_value
JOIN omop.visit_occurrence vo ON p.person_id = vo.person_id
WHERE l."Cirrhosis" = 'Yes, confirmed';
""")

# Ferritin serum 📝
execute("""
INSERT INTO omop.measurement (
    measurement_id, person_id, measurement_concept_id,
    measurement_date, measurement_datetime,
    measurement_type_concept_id, value_as_number,
    unit_concept_id, visit_occurrence_id
)
SELECT nextval('measurement_id_seq'), p.person_id,
    0,  -- TODO: find the LOINC concept ID for Ferritin in serum (search ATHENA: "Ferritin serum")
    vo.visit_start_date, vo.visit_start_date, 32809,
    l."Ferritin serum (ng/mL / µg/L)",
    0,  -- TODO: find the unit concept ID for ng/mL (search ATHENA: "nanogram per milliliter")
    vo.visit_occurrence_id
FROM import.labels l
JOIN omop.person p ON l."Record ID" = p.person_source_value
JOIN omop.visit_occurrence vo ON p.person_id = vo.person_id
WHERE l."Ferritin serum (ng/mL / µg/L)" IS NOT NULL;
""")

# Serum iron ✅
execute("""
INSERT INTO omop.measurement (
    measurement_id, person_id, measurement_concept_id,
    measurement_date, measurement_datetime,
    measurement_type_concept_id, value_as_number,
    unit_concept_id, visit_occurrence_id
)
SELECT nextval('measurement_id_seq'), p.person_id,
    4097596,  -- Serum iron measurement
    vo.visit_start_date, vo.visit_start_date, 32809,
    l."Serum iron (μg/dL)",
    8837,  -- ug/dL
    vo.visit_occurrence_id
FROM import.labels l
JOIN omop.person p ON l."Record ID" = p.person_source_value
JOIN omop.visit_occurrence vo ON p.person_id = vo.person_id
WHERE l."Serum iron (μg/dL)" IS NOT NULL;
""")

query("SELECT value_source_value, measurement_concept_id, unit_concept_id, COUNT(*) AS n FROM omop.measurement GROUP BY 1,2,3 ORDER BY 4 DESC")

### PROCEDURE_OCCURRENCE 📝

Splenectomy is given as a fully worked example. Fill in the concept IDs for Chelation and Hydroxyurea.

| Procedure | concept_id |
|-----------|------------|
| Chelation treatment | ? |
| Hydroxyurea treatment | ? |
| Splenectomy | 2834904 ✅ |

In [ ]:
execute("CREATE SEQUENCE procedure_id_seq START 1;")

# Chelation treatment 📝
execute("""
INSERT INTO omop.procedure_occurrence (
    procedure_occurrence_id, person_id, procedure_concept_id,
    procedure_date, procedure_datetime,
    procedure_end_date, procedure_end_datetime,
    procedure_type_concept_id, visit_occurrence_id, procedure_source_value
)
SELECT nextval('procedure_id_seq'), p.person_id,
    0,  -- TODO: find the concept ID for chelation treatment (search ATHENA: "Chelation")
    vo.visit_start_date, vo.visit_start_date,
    vo.visit_start_date, vo.visit_start_date,
    32865, vo.visit_occurrence_id,
    l."Is the patient on chelation treatment ?"
FROM import.labels l
JOIN omop.person p ON l."Record ID" = p.person_source_value
JOIN omop.visit_occurrence vo ON p.person_id = vo.person_id
WHERE l."Is the patient on chelation treatment ?" = 'Yes';
""")

# Hydroxyurea treatment 📝
execute("""
INSERT INTO omop.procedure_occurrence (
    procedure_occurrence_id, person_id, procedure_concept_id,
    procedure_date, procedure_datetime,
    procedure_end_date, procedure_end_datetime,
    procedure_type_concept_id, visit_occurrence_id, procedure_source_value
)
SELECT nextval('procedure_id_seq'), p.person_id,
    0,  -- TODO: find the concept ID for hydroxyurea therapy (search ATHENA: "Hydroxyurea")
    vo.visit_start_date, vo.visit_start_date,
    vo.visit_start_date, vo.visit_start_date,
    32865, vo.visit_occurrence_id,
    l."Is the patient on hydroxyurea treatment (present time) ?"
FROM import.labels l
JOIN omop.person p ON l."Record ID" = p.person_source_value
JOIN omop.visit_occurrence vo ON p.person_id = vo.person_id
WHERE l."Is the patient on hydroxyurea treatment (present time) ?" = 'Yes';
""")

# Splenectomy ✅ (example)
execute("""
INSERT INTO omop.procedure_occurrence (
    procedure_occurrence_id, person_id, procedure_concept_id,
    procedure_date, procedure_datetime,
    procedure_end_date, procedure_end_datetime,
    procedure_type_concept_id, visit_occurrence_id, procedure_source_value
)
SELECT nextval('procedure_id_seq'), p.person_id,
    2834904,  -- Removal of spleen ✅
    vo.visit_start_date, vo.visit_start_date,
    vo.visit_start_date, vo.visit_start_date,
    32865, vo.visit_occurrence_id,
    l."Has the spleen been removed ?"
FROM import.labels l
JOIN omop.person p ON l."Record ID" = p.person_source_value
JOIN omop.visit_occurrence vo ON p.person_id = vo.person_id
WHERE l."Has the spleen been removed ?" = 'Yes, totally';
""")

query("SELECT procedure_source_value, procedure_concept_id, COUNT(*) AS n FROM omop.procedure_occurrence GROUP BY 1, 2 ORDER BY 3 DESC")

---
## Final Overview

Run this query to see the row count for every table in your database. Use this to verify your work.

**Expected non-zero tables after completing all TODOs:** person, observation_period, visit_occurrence, condition_occurrence, observation, measurement, procedure_occurrence

In [ ]:
query("""
SELECT table_schema, table_name,
       (xpath('/row/cnt/text()',
              query_to_xml(format('SELECT COUNT(*) AS cnt FROM %I.%I', table_schema, table_name),
                           false, true, '')))[1]::text::integer AS row_count
FROM information_schema.tables
WHERE table_schema NOT IN ('pg_catalog', 'information_schema')
  AND table_type = 'BASE TABLE'
ORDER BY table_schema, table_name
""")

---
## Bonus: Query Your OMOP Database

Now that your database is populated, try answering some clinical questions with SQL.

### Question 1: How many patients have each diagnosis?

In [ ]:
query("""
SELECT
    c.condition_source_value AS diagnosis,
    COUNT(DISTINCT c.person_id) AS n_patients
FROM omop.condition_occurrence c
JOIN omop.person p ON p.person_id = c.person_id
WHERE c.condition_source_value IN ('Alpha-thalassemia', 'Beta-thalassemia', 'Sickle cell disease')
GROUP BY 1
ORDER BY 2 DESC;
""")

### Question 2: What is the median ferritin level, split by sex?

In [ ]:
query("""
SELECT
    p.gender_source_value AS sex,
    ROUND(PERCENTILE_CONT(0.5) WITHIN GROUP (ORDER BY m.value_as_number)::numeric, 1) AS median_ferritin_ngml,
    COUNT(*) AS n_measurements
FROM omop.measurement m
JOIN omop.person p ON p.person_id = m.person_id
WHERE m.value_source_value = 'Ferritin serum (ng/mL / µg/L)'  -- or use measurement_concept_id once filled in
  AND m.value_as_number IS NOT NULL
GROUP BY 1;
""")

### Question 3: What proportion of patients are on chelation treatment? (Write your own query)

Hint: count distinct `person_id` in `procedure_occurrence` where the procedure matches chelation, divide by total persons.

In [ ]:
# Write your query here
query("""
-- TODO: write a query that returns the proportion of patients on chelation treatment
SELECT 'your query here' AS result
""")